# Text-to-SQL T5-small Baseline


Task:

```text
question + table columns -> SQL query
```

## 1. Install

In [ ]:
!pip -q install -U transformers accelerate sentencepiece scikit-learn
!pip -q install "datasets<4.0.0" "pandas==2.2.2"

## 2. Imports

In [ ]:
import random
import re
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

## 3. Config

In [ ]:
MODEL_NAME = "google-t5/t5-small"
DATASET_NAME = "Salesforce/wikisql"

USE_FULL_TRAIN = True
TRAIN_SIZE = 10000

VALID_EVAL_SIZE = 1000
TEST_EVAL_SIZE = 1000

MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 128

BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4
EPOCHS = 2
LEARNING_RATE = 3e-4
SEED = 42

OUTPUT_DIR = "./text2sql-t5-small-baseline"

## 4. Setup

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

if device == "cuda":
    print(torch.cuda.get_device_name(0))

## 5. Load WikiSQL

In [ ]:
dataset = load_dataset(DATASET_NAME, trust_remote_code=True)
dataset

## 6. Serialization functions

In [ ]:
AGG_OPS = ["", "MAX", "MIN", "COUNT", "SUM", "AVG"]
COND_OPS = ["=", ">", "<", "OP"]


def clean_text(x):
    x = str(x).replace("\n", " ")
    x = re.sub(r"\s+", " ", x)
    return x.strip()


def get_columns(example):
    table = example["table"]

    if "header" in table:
        return table["header"]
    if "headers" in table:
        return table["headers"]
    if "column_names" in table:
        return table["column_names"]

    raise ValueError(f"Unknown table format: {table.keys()}")


def serialize_input(example):
    question = clean_text(example["question"])
    columns = " | ".join(clean_text(c) for c in get_columns(example))
    return f"translate to SQL: question: {question} columns: {columns}"


def iter_conditions(conditions):
    if isinstance(conditions, dict):
        col_indices = conditions.get("column_index", [])
        op_indices = conditions.get("operator_index", [])
        values = conditions.get("condition", [])

        for col_idx, op_idx, value in zip(col_indices, op_indices, values):
            yield col_idx, op_idx, value
    else:
        for cond in conditions:
            yield cond[0], cond[1], cond[2]


def serialize_sql(example):
    sql = example["sql"]
    columns = get_columns(example)

    select_col = clean_text(columns[sql["sel"]])
    agg = AGG_OPS[sql["agg"]]

    if agg:
        query = f"SELECT {agg}({select_col}) FROM table"
    else:
        query = f"SELECT {select_col} FROM table"

    where_parts = []

    for col_idx, op_idx, value in iter_conditions(sql.get("conds", [])):
        col_name = clean_text(columns[col_idx])
        op = COND_OPS[op_idx] if op_idx < len(COND_OPS) else "="
        value = clean_text(value)
        where_parts.append(f"{col_name} {op} '{value}'")

    if where_parts:
        query += " WHERE " + " AND ".join(where_parts)

    return query

## 7. Check preprocessing

In [ ]:
for i in range(3):
    ex = dataset["train"][i]
    print("QUESTION:", ex["question"])
    print("INPUT:", serialize_input(ex))
    print("TARGET:", serialize_sql(ex))
    print("-" * 80)

## 8. Prepare data

In [ ]:
if USE_FULL_TRAIN:
    train_raw = dataset["train"].shuffle(seed=SEED)
else:
    train_raw = dataset["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))

valid_raw = dataset["validation"].shuffle(seed=SEED).select(range(VALID_EVAL_SIZE))
test_raw = dataset["test"].shuffle(seed=SEED).select(range(TEST_EVAL_SIZE))

print(len(train_raw), len(valid_raw), len(test_raw))

In [ ]:
def add_text_fields(example):
    return {
        "input_text": serialize_input(example),
        "target_text": serialize_sql(example)
    }

train_text = train_raw.map(add_text_fields)
valid_text = valid_raw.map(add_text_fields)
test_text = test_raw.map(add_text_fields)

pd.DataFrame(train_text[:3])[["input_text", "target_text"]]

## 9. Tokenizer and model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.to(device)

## 10. Tokenize

In [ ]:
def preprocess(batch):
    inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    inputs["labels"] = labels["input_ids"]
    return inputs


remove_cols = train_text.column_names

train_ds = train_text.map(preprocess, batched=True, remove_columns=remove_cols)
valid_ds = valid_text.map(preprocess, batched=True, remove_columns=remove_cols)
test_ds = test_text.map(preprocess, batched=True, remove_columns=remove_cols)

## 11. Normalized exact match

In [ ]:
def normalize_sql(sql):
    sql = sql.lower()
    sql = sql.replace('"', "'")
    sql = re.sub(r"\s+", " ", sql)
    sql = re.sub(r"\s*([(),=<>])\s*", r"\1", sql)
    return sql.strip()


def normalized_exact_match(preds, refs):
    return np.mean([
        normalize_sql(pred) == normalize_sql(ref)
        for pred, ref in zip(preds, refs)
    ])

## 12. Train

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    save_total_limit=2,
    report_to="none",
    seed=SEED
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator
)

trainer.train()

## 13. Generate

In [ ]:
def generate_sql(input_text, num_beams=4):
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_length=MAX_TARGET_LENGTH,
            num_beams=num_beams
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


for i in range(5):
    inp = test_text[i]["input_text"]
    tgt = test_text[i]["target_text"]
    pred = generate_sql(inp)

    print("INPUT:", inp)
    print("TARGET:", tgt)
    print("PRED:", pred)
    print("MATCH:", normalize_sql(pred) == normalize_sql(tgt))
    print("-" * 80)

## 14. Test subset evaluation

In [ ]:
def generate_many(input_texts, batch_size=16):
    preds = []

    for i in range(0, len(input_texts), batch_size):
        batch = input_texts[i:i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LENGTH
        )

        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        model.eval()
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_length=MAX_TARGET_LENGTH,
                num_beams=4
            )

        preds.extend(tokenizer.batch_decode(output_ids, skip_special_tokens=True))

    return preds


test_inputs = test_text["input_text"]
test_targets = test_text["target_text"]
test_preds = generate_many(test_inputs)

exact_match = normalized_exact_match(test_preds, test_targets)
exact_match

## 15. Error analysis

In [ ]:
results = pd.DataFrame({
    "input": test_inputs,
    "target": test_targets,
    "prediction": test_preds
})

results["correct"] = [
    normalize_sql(pred) == normalize_sql(target)
    for pred, target in zip(results["prediction"], results["target"])
]

results["correct"].value_counts(normalize=True)

In [ ]:
results[results["correct"] == False][["input", "target", "prediction"]].head(20)

## 16. Save

In [ ]:
trainer.save_model("./text2sql-t5-small-baseline-best")
tokenizer.save_pretrained("./text2sql-t5-small-baseline-best")

In [ ]:
baseline_summary = {
    "model": "google-t5/t5-small",
    "training_data": "full WikiSQL train split, 56,355 examples",
    "evaluation_data": "1,000 sampled WikiSQL test examples",
    "metric": "normalized exact match",
    "result": 0.594,
    "epochs": 2,
    "batch_size": 8,
    "gradient_accumulation_steps": 4,
    "effective_batch_size": 32,
    "gpu": "Tesla T4"
}

baseline_summary